<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 11 — Flow Matching for Synthetic Floorplan Generation (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 11**.

It is adapted from your uploaded reference notebook **`MLM9_Flow_Matching_Receipt_Generation.ipynb`**, but rewritten for the actual assignment task: **train a flow matching model on floorplan images and generate new synthetic floorplans**.

## What Task 11 asks you to do
- use the notebook **`MLM9 - Flow Matching - Receipt Generation.ipynb`** as context
- download the zip file **`floorplans_v2-20251223T170650Z-3-001.zip`**
- train a **flow matching model** on the floorplan images
- generate new **synthetic floorplans**

## What this notebook does
1. uploads or locates the floorplans zip
2. extracts the images
3. builds a PyTorch dataset
4. trains a lightweight flow-matching U-Net
5. plots the training loss
6. generates synthetic floorplans
7. saves the checkpoint and generated images

## Colab recommendation
Use **Runtime → Change runtime type → T4 GPU** before training.

## Beginner note
Run the cells **from top to bottom** in order.
Do not skip cells.


## Step 0 — Install packages

Most of these are already available in Colab, but this cell makes sure everything is ready.


In [ ]:
!pip -q install pillow tqdm matplotlib

## Step 1 — Imports and basic setup


In [ ]:
import os
import zipfile
import random
import math
import glob
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

## Step 2 — Set a random seed

This helps make the run more reproducible.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✅ Seed set.")

## Step 3 — Hyperparameters

These defaults are chosen to be more **Colab-friendly** than a large model.

### Recommended defaults
- `IMG_SIZE = 128`
- `BATCH_SIZE = 16`
- `EPOCHS = 12`

If training is too slow, reduce:
- `IMG_SIZE` to `96`
- `BATCH_SIZE` to `8`
- `EPOCHS` to `8`

If your GPU memory is fine and you want better quality, increase `EPOCHS` later.


In [ ]:
CONFIG = {
    "IMG_SIZE": 128,
    "CHANNELS": 1,              # 1 = grayscale, faster for Colab
    "BATCH_SIZE": 16,
    "LR": 1e-4,
    "EPOCHS": 12,
    "INFERENCE_STEPS": 80,
    "BASE_CHANNELS": 32,
    "NUM_WORKERS": 2,
    "MAX_IMAGES": None,         # set to an integer like 2000 if dataset is extremely large
    "PREVIEW_EVERY": 4,
    "GENERATE_N": 12,
    "OUTPUT_DIR": "task11_outputs"
}

COLOR_MODE = "L" if CONFIG["CHANNELS"] == 1 else "RGB"

Path(CONFIG["OUTPUT_DIR"]).mkdir(exist_ok=True)

print(CONFIG)

## Step 4 — Upload the floorplans zip

### What to upload
Upload the assignment file:

`floorplans_v2-20251223T170650Z-3-001.zip`

If your zip is already in Colab, you can skip the upload step and just set `ZIP_PATH` manually in the next cell.


In [ ]:
ZIP_PATH = "/content/floorplans_v2-20251223T170650Z-3-001.zip"

if os.path.exists(ZIP_PATH):
    print("✅ Found zip already at:", ZIP_PATH)
else:
    print("Zip not found at the default path.")
    print("Run the next upload cell if needed.")

In [ ]:
# Run this only if the zip is not already present in /content
from google.colab import files

uploaded = files.upload()

if len(uploaded) > 0:
    uploaded_name = list(uploaded.keys())[0]
    ZIP_PATH = f"/content/{uploaded_name}"
    print("✅ Uploaded zip path:", ZIP_PATH)
else:
    print("No file uploaded in this run.")

## Step 5 — Extract the zip file


In [ ]:
EXTRACT_DIR = "/content/floorplan_data"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Could not find zip file at: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("✅ Extracted to:", EXTRACT_DIR)

## Step 6 — Find all image files recursively

The zip may contain nested folders, so we search through everything.


In [ ]:
def find_image_files(root_dir):
    image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
    image_paths = []

    for current_root, _, files in os.walk(root_dir):
        for fname in files:
            ext = Path(fname).suffix.lower()
            if ext in image_extensions:
                image_paths.append(os.path.join(current_root, fname))

    image_paths = sorted(image_paths)
    return image_paths

all_image_paths = find_image_files(EXTRACT_DIR)

if CONFIG["MAX_IMAGES"] is not None:
    all_image_paths = all_image_paths[:CONFIG["MAX_IMAGES"]]

print("Number of floorplan images found:", len(all_image_paths))

if len(all_image_paths) == 0:
    raise RuntimeError("No image files were found after extraction. Check the zip contents.")

## Step 7 — Preview a few real floorplans


In [ ]:
def show_image_grid(image_paths, n=6, cols=3, title="Sample floorplans"):
    n = min(n, len(image_paths))
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i in range(n):
        img = Image.open(image_paths[i]).convert(COLOR_MODE)
        plt.subplot(rows, cols, i + 1)
        if CONFIG["CHANNELS"] == 1:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
        plt.title(Path(image_paths[i]).name[:30])
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_image_grid(all_image_paths, n=6, cols=3)

## Step 8 — Create the dataset and dataloader

We convert images to:
- grayscale (for faster training)
- fixed size (`IMG_SIZE x IMG_SIZE`)
- tensors in `[0, 1]`


In [ ]:
class FloorplanDataset(Dataset):
    def __init__(self, image_paths, img_size=128, channels=1):
        self.image_paths = image_paths
        self.channels = channels
        color_mode = "L" if channels == 1 else "RGB"

        transform_list = [
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ]

        self.transform = transforms.Compose(transform_list)
        self.color_mode = color_mode

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert(self.color_mode)
        image = self.transform(image)
        return image

dataset = FloorplanDataset(
    image_paths=all_image_paths,
    img_size=CONFIG["IMG_SIZE"],
    channels=CONFIG["CHANNELS"]
)

train_loader = DataLoader(
    dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CONFIG["NUM_WORKERS"],
    pin_memory=(device.type == "cuda"),
    drop_last=True
)

print("Dataset size:", len(dataset))
print("Batches per epoch:", len(train_loader))

## Step 9 — Define the flow matching model

This is a lightweight U-Net-like model with time embeddings.
It is much smaller than full research-scale models, which makes it more suitable for Google Colab.


In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half_dim = self.dim // 2
        device = time.device
        scale = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -scale)
        emb = time[:, None] * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, groups=8):
        super().__init__()
        self.time_proj = nn.Linear(time_dim, out_ch)

        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.act = nn.SiLU()

        if in_ch != out_ch:
            self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        else:
            self.skip = nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = self.act(h)

        t = self.time_proj(t_emb)[:, :, None, None]
        h = h + t

        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)

        return h + self.skip(x)


class TinyFlowUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=32, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4

        self.in_conv = nn.Conv2d(in_channels, c1, kernel_size=3, padding=1)

        self.down1 = ResBlock(c1, c1, time_dim)
        self.pool1 = nn.Conv2d(c1, c2, kernel_size=4, stride=2, padding=1)

        self.down2 = ResBlock(c2, c2, time_dim)
        self.pool2 = nn.Conv2d(c2, c3, kernel_size=4, stride=2, padding=1)

        self.mid = ResBlock(c3, c3, time_dim)

        self.up2 = nn.ConvTranspose2d(c3, c2, kernel_size=4, stride=2, padding=1)
        self.up_block2 = ResBlock(c2 + c2, c2, time_dim)

        self.up1 = nn.ConvTranspose2d(c2, c1, kernel_size=4, stride=2, padding=1)
        self.up_block1 = ResBlock(c1 + c1, c1, time_dim)

        self.out = nn.Conv2d(c1, in_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)

        x0 = self.in_conv(x)
        x1 = self.down1(x0, t_emb)
        x2 = self.pool1(x1)

        x3 = self.down2(x2, t_emb)
        x4 = self.pool2(x3)

        xm = self.mid(x4, t_emb)

        xu = self.up2(xm)
        xu = torch.cat([xu, x3], dim=1)
        xu = self.up_block2(xu, t_emb)

        xu = self.up1(xu)
        xu = torch.cat([xu, x1], dim=1)
        xu = self.up_block1(xu, t_emb)

        return self.out(xu)


class FlowMatching:
    def compute_loss(self, model, x_data):
        batch_size = x_data.shape[0]

        x_noise = torch.randn_like(x_data)
        t = torch.rand(batch_size, device=x_data.device)

        t_view = t[:, None, None, None]
        x_t = (1.0 - t_view) * x_noise + t_view * x_data

        target_velocity = x_data - x_noise
        pred_velocity = model(x_t, t)

        return F.mse_loss(pred_velocity, target_velocity)

    @torch.no_grad()
    def sample(self, model, n_samples, size, channels, steps=80):
        model.eval()

        x = torch.randn(n_samples, channels, size, size, device=device)
        dt = 1.0 / steps

        for i in range(steps):
            t = torch.full((n_samples,), i / steps, device=device)
            v = model(x, t)
            x = x + dt * v

        model.train()
        return x

model = TinyFlowUNet(
    in_channels=CONFIG["CHANNELS"],
    base_channels=CONFIG["BASE_CHANNELS"]
).to(device)

flow = FlowMatching()

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model created with {total_params:,} parameters")

## Step 10 — Helper functions for visualization and saving


In [ ]:
def show_tensor_grid(tensor_batch, nrow=4, title="Generated samples"):
    grid = make_grid(tensor_batch, nrow=nrow, normalize=True, scale_each=True)
    grid_np = grid.detach().cpu().permute(1, 2, 0).numpy()

    plt.figure(figsize=(10, 10))
    if CONFIG["CHANNELS"] == 1:
        plt.imshow(grid_np.squeeze(), cmap="gray")
    else:
        plt.imshow(grid_np)
    plt.title(title)
    plt.axis("off")
    plt.show()


def save_generated_grid(tensor_batch, file_path, nrow=4):
    save_image(tensor_batch, file_path, nrow=nrow, normalize=True, scale_each=True)


def save_individual_samples(tensor_batch, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for i, sample in enumerate(tensor_batch):
        save_image(sample, f"{out_dir}/sample_{i+1:02d}.png", normalize=True, scale_each=True)


def save_checkpoint(model, optimizer, history, file_path):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "config": CONFIG
    }
    torch.save(checkpoint, file_path)
    print("✅ Checkpoint saved to:", file_path)

## Step 11 — Train the flow matching model

This is the main training cell.

### What it does
- uses Adam optimizer
- uses mixed precision on GPU when available
- tracks epoch loss
- periodically generates preview samples

### Note
The first epoch may be slower in Colab.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

loss_history = []
preview_snapshots = []

for epoch in range(CONFIG["EPOCHS"]):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}")
    for batch in pbar:
        images = batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            loss = flow.compute_loss(model, images)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

    epoch_loss = running_loss / max(len(train_loader), 1)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch+1} average loss: {epoch_loss:.6f}")

    if (epoch + 1) % CONFIG["PREVIEW_EVERY"] == 0 or (epoch + 1) == CONFIG["EPOCHS"]:
        with torch.no_grad():
            preview = flow.sample(
                model=model,
                n_samples=4,
                size=CONFIG["IMG_SIZE"],
                channels=CONFIG["CHANNELS"],
                steps=CONFIG["INFERENCE_STEPS"]
            )
        preview_snapshots.append((epoch + 1, preview.detach().cpu()))
        show_tensor_grid(preview, nrow=2, title=f"Preview after epoch {epoch+1}")

print("✅ Training complete.")

## Step 12 — Plot the training loss


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average training loss")
plt.title("Flow Matching Training Loss")
plt.grid(True)
plt.show()

## Step 13 — Save the trained checkpoint


In [ ]:
checkpoint_path = f"{CONFIG['OUTPUT_DIR']}/task11_flow_matching_floorplans_checkpoint.pth"
save_checkpoint(model, optimizer, loss_history, checkpoint_path)

## Step 14 — Generate synthetic floorplans

This uses the trained model to generate a larger batch of new floorplans.


In [ ]:
with torch.no_grad():
    generated_samples = flow.sample(
        model=model,
        n_samples=CONFIG["GENERATE_N"],
        size=CONFIG["IMG_SIZE"],
        channels=CONFIG["CHANNELS"],
        steps=CONFIG["INFERENCE_STEPS"]
    )

show_tensor_grid(generated_samples, nrow=4, title="Synthetic Floorplans")

## Step 15 — Save the generated floorplans


In [ ]:
grid_path = f"{CONFIG['OUTPUT_DIR']}/synthetic_floorplans_grid.png"
single_dir = f"{CONFIG['OUTPUT_DIR']}/synthetic_floorplans"

save_generated_grid(generated_samples, grid_path, nrow=4)
save_individual_samples(generated_samples, single_dir)

print("✅ Saved generated outputs:")
print("-", grid_path)
print("-", single_dir)

## Step 16 — Save the training loss values to CSV


In [ ]:
import csv

loss_csv_path = f"{CONFIG['OUTPUT_DIR']}/training_loss.csv"
with open(loss_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "avg_train_loss"])
    for i, value in enumerate(loss_history, start=1):
        writer.writerow([i, value])

print("✅ Saved loss history to:", loss_csv_path)

## Step 17 — Create a short summary text file

This helps if you want to submit a brief note with your notebook outputs.


In [ ]:
summary_text = f"""
Task 11 Summary

Dataset:
- Number of floorplan images used: {len(dataset)}
- Image size: {CONFIG['IMG_SIZE']} x {CONFIG['IMG_SIZE']}
- Channels: {CONFIG['CHANNELS']}

Training:
- Epochs: {CONFIG['EPOCHS']}
- Batch size: {CONFIG['BATCH_SIZE']}
- Learning rate: {CONFIG['LR']}
- Final average training loss: {loss_history[-1] if len(loss_history) > 0 else 'N/A'}

Generation:
- Number of synthetic floorplans generated: {CONFIG['GENERATE_N']}
- Inference steps: {CONFIG['INFERENCE_STEPS']}

Main output files:
- {checkpoint_path}
- {grid_path}
- {loss_csv_path}
""".strip()

summary_path = f"{CONFIG['OUTPUT_DIR']}/task11_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("\n✅ Saved summary to:", summary_path)

## Step 18 — Zip the outputs folder


In [ ]:
archive_base = "task11_outputs_archive"
archive_path = shutil.make_archive(archive_base, "zip", CONFIG["OUTPUT_DIR"])
print("✅ Created archive:", archive_path)

## Step 19 — Download the key outputs

Run this after training finishes.


In [ ]:
from google.colab import files

files.download(checkpoint_path)
files.download(grid_path)
files.download(loss_csv_path)
files.download(summary_path)
files.download(archive_path)

## Step 20 — Optional: load the checkpoint later and generate again

Use this if your runtime disconnects and you want to regenerate samples from a saved checkpoint.


In [ ]:
def load_checkpoint_and_generate(checkpoint_path, n_samples=8):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    ckpt = torch.load(checkpoint_path, map_location=device)

    loaded_model = TinyFlowUNet(
        in_channels=ckpt["config"]["CHANNELS"],
        base_channels=ckpt["config"]["BASE_CHANNELS"]
    ).to(device)

    loaded_model.load_state_dict(ckpt["model_state_dict"])
    loaded_model.eval()

    loaded_flow = FlowMatching()

    with torch.no_grad():
        samples = loaded_flow.sample(
            model=loaded_model,
            n_samples=n_samples,
            size=ckpt["config"]["IMG_SIZE"],
            channels=ckpt["config"]["CHANNELS"],
            steps=ckpt["config"]["INFERENCE_STEPS"]
        )

    show_tensor_grid(samples, nrow=4, title="Samples from loaded checkpoint")
    return samples

# Example:
# loaded_samples = load_checkpoint_and_generate(checkpoint_path, n_samples=8)

## Troubleshooting

### If no images are found
The zip may contain nested folders. This notebook already searches recursively, but if it still finds zero images:
- inspect the extracted folder manually
- verify the image files are `.png`, `.jpg`, `.jpeg`, `.bmp`, or `.webp`

### If training is too slow
Reduce:
```python
CONFIG["IMG_SIZE"] = 96
CONFIG["BATCH_SIZE"] = 8
CONFIG["EPOCHS"] = 8
```

### If Colab runs out of memory
Reduce:
```python
CONFIG["BATCH_SIZE"] = 8
CONFIG["BASE_CHANNELS"] = 16
CONFIG["IMG_SIZE"] = 96
```

### If outputs look noisy
That is normal for a small model trained briefly.
For better samples, try:
- more epochs
- more inference steps
- a slightly larger image size
- a larger dataset subset

### If you want color floorplans instead of grayscale
Change:
```python
CONFIG["CHANNELS"] = 3
```
Then rerun the notebook from the config cell onward.
